# Markovitz' mean-variance-modell  

##### Målet er å finne en optimal portefølje ved hjelp av Markowitz mean-variance modell. jeg vil lese inn historisk prisdata, beregne avkastning og beregne forventet avkastning og risiko. lage en covariance matrise og simulere ulike kombinasjoner for og optimere for å finne portefølje vektene som gir best forhold mellom avkastning og risiko. 

##### jeg vil også gjøre en analyse på egenskapene i avkastningsereiene til de ulike ETF-ene for å gi mere kontekts til optimaliseringen dette er det første jeg fortar meg i oppgaven.

##### Det er en portefølje begrensning på 5 assets. begrensningen ligger i hovedsak i skrivingen til excel som ikke er gjort dynamisk.



In [ ]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import math
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter
from IPython.display import display
from scipy.stats import norm, kurtosis, skew

base_dir = os.getcwd()

ExcelOut=base_dir


In [87]:
#   Farger for ploting
bar_color='#0c2a1e'
grid_color='#777777'

In [ ]:
#   Lesing og enkle beregninger av data
df=pd.read_csv('ETF_price_volume.csv')
df['mid']=(df['Bid']+df['Ask'])/2
df['abs_spread']=df['Ask']-df['Bid']
df['pct_spread']=df['abs_spread']/df['mid']

print(df.head())

tickers=df['Ticker'].unique().tolist()

antall_assets=df['Ticker'].nunique()
print(antall_assets)

print(tickers)

#   kostnad csv
df_cost=pd.read_csv('ETF_cost.csv')
print(df_cost)

In [89]:
#   Funksjoner brukt i koden
def ret(PX_LAST):
    priser=np.array(PX_LAST)
    daglig_avkastning=priser[1:]/priser[:-1]-1
    return daglig_avkastning

def portefølje_avkastning(vekter,mu_vektor):
    return np.dot(vekter,mu_vektor)

def portefølje_risk(vekt,covariansmatrise):
    return np.sqrt(vekt.T @covariansmatrise @vekt)

def portefølje_kostnad(opti_vekter_df):
    
    løpende_porteføljekostnad = (
    df_cost[['Ticker','fund_expense_ratio']]
    .merge(
        opti_vekter_df[['Ticker','Optimal vekt']],
        on='Ticker',
        how='inner'
    ))
    løpende_porteføljekostnad['kostnad']= løpende_porteføljekostnad['Optimal vekt']*løpende_porteføljekostnad['fund_expense_ratio']
    Total_løpende_port_kostnad=løpende_porteføljekostnad['kostnad'].sum()

    Forventet_transaksjonskostnad=(mid_spread[['Ticker','Gjennomsnittlig prosent spread']]
                                   .merge(opti_vekter_df[['Ticker','Optimal vekt']],
                                          on='Ticker',
                                          how='inner'
                                          ))
    
    Forventet_transaksjonskostnad['kostnad']=Forventet_transaksjonskostnad['Optimal vekt']*Forventet_transaksjonskostnad['Gjennomsnittlig prosent spread']
    Total_forventet_transaksjonskostnad=Forventet_transaksjonskostnad['kostnad'].sum()

    return  Total_løpende_port_kostnad, Total_forventet_transaksjonskostnad

In [ ]:
#   Portefølje Beregninger
avkastning_df=pd.DataFrame()

for ticker in tickers:
    sort=df['Ticker']==ticker
    sortering=df[sort].copy()
    sortering=sortering.sort_values('Date')
    avkastning=ret(sortering['PX_LAST'])
    
    datoer = sortering['Date'].iloc[1:].values

    temp_df=pd.DataFrame({
        'Date': datoer, 
        ticker:avkastning
        })

    if avkastning_df.empty:
        avkastning_df=temp_df
    else:
        avkastning_df=pd.merge(avkastning_df,temp_df, on= 'Date', how='outer')

# print(avkastning_df)

stats_df = pd.DataFrame()

#   egenskaper i avakstningseriene
for ticker in tickers:
    stats = avkastning_df[ticker].describe()
    stats_df[ticker] = stats

#   Bygging av den optimale porteføljen

ret_u=avkastning_df[tickers].dropna()

# forvantet daglig avkastning og Standard avvik for hver ETF
mu=ret_u.mean()
mu_annualisert=mu*252
mu_annualisert_df=mu_annualisert.to_frame(name='Annualisert avkastning')
mu_annualisert_df['Annualisert avkastning']=mu_annualisert_df['Annualisert avkastning'].map(lambda x: f'{x:.2%}')
sigma=ret_u.std()
sigma_annualisert=sigma*np.sqrt(252)
sigma_annualisert_df=sigma_annualisert.to_frame(name='Annualisert standaravvik')
sigma_annualisert_df['Annualisert standaravvik']=sigma_annualisert_df['Annualisert standaravvik'].map(lambda x: f'{x:.2%}')
print('------forventet daglig avkastning----------')
print(mu)
print('------Annualisert forventet årlig avkastning---------')
print(mu_annualisert)
print('-----forventet daglig Standard avvik-------')
print(sigma)
print('------Forventet annualisert standard avvik--------')
print(sigma_annualisert)
# covariance matrise
covariansmatrise=ret_u.cov()
annaualisert_covariance=covariansmatrise*252
print('-----covariance matrise--------')
print(covariansmatrise)
print('-----Annualisert covariance--------')
print(annaualisert_covariance)

#   simulering av porteføljer

mu_vektor=mu_annualisert.to_numpy()
cov_matrise=annaualisert_covariance.to_numpy()

antall_simulerings_porteføljer=10000 #  antall simuleringer


sim_portefølje_avkastning=np.zeros(antall_simulerings_porteføljer) #lager matriser med 0 til bruk for bergeninger
sim_portefølje_risk=np.zeros(antall_simulerings_porteføljer)
sim_sharp=np.zeros(antall_simulerings_porteføljer)
sim_vekter=np.zeros((antall_simulerings_porteføljer,antall_assets))

risikofri_rente=0.042   #   risikofri rente blir brukt for å beregne Sharpe ratio

for i in range(antall_simulerings_porteføljer):
    vekter=np.random.random(antall_assets)
    vekter=vekter/np.sum(vekter)

    sim_vekter[i,:]=vekter

    port_avkastning=portefølje_avkastning(vekter,mu_vektor)
    port_risk=portefølje_risk(vekter,cov_matrise)
    sharpe=(port_avkastning-risikofri_rente)/port_risk

    sim_portefølje_avkastning[i]=port_avkastning
    sim_portefølje_risk[i]=port_risk
    sim_sharp[i]= sharpe

maks_sharpe=np.argmax(sim_sharp)

opti_vekter=sim_vekter[maks_sharpe]
opti_avkastning=sim_portefølje_avkastning[maks_sharpe]
opti_risk=sim_portefølje_risk[maks_sharpe]
opti_shapre=sim_sharp[maks_sharpe]


sim_stat={'Portefølje Stats':['Forventet annualisert avkastning','Forventet annualisert risiko','forvantet Sharpe ration'],
          'Forventning':[f'{opti_avkastning:.2%}', f'{opti_risk: .2%}', f'{opti_shapre: .2f}']
          }

opti_av_risk_sharp_df=pd.DataFrame(sim_stat)

print('------Optimal portefølje basert på maks sharpeRatio----------------')

for ticker,vekt in zip(tickers,opti_vekter):
    print(f'{ticker}: {vekt:.2%}')

print(f'Forventet annualisert avkastning: {opti_avkastning:.2%}')
print(f'forventet annualisert risiko: {opti_risk: .2%}')
print(f'Sharpe ration: {opti_shapre: .2f}')

sort_ind=np.argsort(sim_portefølje_risk)
sortert_risk=sim_portefølje_risk[sort_ind]
sortert_avkastning=sim_portefølje_avkastning[sort_ind]

sim_resultater_df = pd.DataFrame(sim_vekter, columns=tickers)

sim_resultater_df['Forventet avkastning'] = sim_portefølje_avkastning
sim_resultater_df['Risiko'] = sim_portefølje_risk
sim_resultater_df['Sharpe'] = sim_sharp

print('---- simulerings resultat------')
print(sim_resultater_df.head())

print('----- Egenskaper i Avkastningseriene--------')
display(stats_df)

In [ ]:
# Plotting av grafer for å visualisere bergeningen og optimaliseringen av porteføljen

antall_kolonner=2
antall_rader=math.ceil((antall_assets)/antall_kolonner)

#   Ploting av Histogram for daglig av kastning egenskaper knyttet til seriene
fig2, axes = plt.subplots(antall_rader, antall_kolonner, figsize=(20, 12))
plt.subplots_adjust(hspace=0.4)
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    axes[i].hist(avkastning_df[ticker].dropna(),bins=100,density=True, alpha=0.6, color=bar_color, edgecolor='black')
    axes[i].set_title(ticker)
    axes[i].grid(color=grid_color, linestyle='--', linewidth=0.5, alpha=0.7)
    skewness = skew(avkastning_df[ticker].dropna())
    kurt_val = kurtosis(avkastning_df[ticker].dropna())
    stats_text = (f'{ticker}\nSkewness: {skewness:.4f}\nKurtosis: {kurt_val:.4f}')
    axes[i].text(0.19, 0.65, stats_text,transform=axes[i].transAxes, color=bar_color, fontsize=10)

for j in range(antall_assets, len(axes)):
    fig2.delaxes(axes[j])

plot_path_histogram_avkastning=os.path.join(ExcelOut, 'histogram_avkastning.png')
plt.savefig(plot_path_histogram_avkastning,dpi=200,bbox_inches='tight')


#   Plotting av efficient frontier ie. utfallsrommet for porteføljen
Frontier_r=[]
frontier_avk=[]

maks_avkastning_ht=-np.inf

for a, avkastning in zip(sortert_risk,sortert_avkastning):
    if avkastning>maks_avkastning_ht:
        Frontier_r.append(a)
        frontier_avk.append(avkastning)
        maks_avkastning_ht=avkastning

plt.figure(figsize=(16, 9))

plt.plot(Frontier_r,frontier_avk, linewidth=2,label='Efficient Frontier', color=bar_color)

scatter = plt.scatter(
    sim_portefølje_risk,
    sim_portefølje_avkastning,
    c=sim_sharp,
    alpha=0.6,
)

plt.scatter(
    opti_risk,
    opti_avkastning,
    marker='*',
    s=300,
    color='r',
    label='Optimal portefølje'
)
plot_path_efficientfront=os.path.join(ExcelOut, 'EfficientFront.png')

plt.title(f'{antall_simulerings_porteføljer} Simulerte porteføljer og optimal portefølje')
plt.xlabel('Annualisert risiko')
plt.ylabel('Annualisert avkastning')
plt.legend()
plt.grid(True)
plt.savefig(plot_path_efficientfront, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
#   kostnads beregninger og diverse hjelpe beregninger før skriveing av resultat til Excel

# kostnads analyse
spread=[]

mid_spread=pd.DataFrame()
for ticker in tickers:
    sort=df['Ticker']==ticker
    mid_df=df[sort].copy()
    s_mid=mid_df['pct_spread'].mean()

    spread.append({'Ticker': ticker,
                  'Gjennomsnittlig prosent spread': s_mid
                  })

mid_spread=pd.DataFrame(spread)

#   Skrive om numpy arrys til DataFrame

opti_vekter_df = pd.DataFrame({
    'Ticker': tickers,
    'Optimal vekt': opti_vekter
})
opti_vekter_df['Optimal vekt %'] = opti_vekter_df['Optimal vekt'].map(lambda x: f'{x:.2%}')


#   Beregnign av porteføljekostnader

Løpende_kostnader, Transaksjonskostnader=portefølje_kostnad(opti_vekter_df)

kost={'Type':['Løpende kostnader','Transaksjonskostnader'],
      'kostnad':[Løpende_kostnader,Transaksjonskostnader*100]}

kostnad_df=pd.DataFrame(kost)                        

print('------Kostnader--------')
print('Løpende', Løpende_kostnader)
print('Transkajsonskostnad',Transaksjonskostnader*100)

In [ ]:
#   Skriving til Excel  

sheets=['OptimalPortefølje', 'avkastnings_analyse', 'Kostnadsanalyse']

output_excel_path = os.path.join(ExcelOut, f"OptimalPortefølje_analyse_resultat.xlsx")

with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    for s in sheets:
        if s== 'OptimalPortefølje':
            sheetname = s
            opti_vekter_df.to_excel(writer, sheet_name=sheetname, index=False)
            opti_av_risk_sharp_df.to_excel(writer,sheet_name=sheetname,startrow=9,startcol=0,index=False)

        elif s=='avkastnings_analyse':
                sheetname = s
                stats_df.to_excel(writer,sheet_name=sheetname)
                mu_annualisert_df.to_excel(writer,sheet_name=sheetname,startrow=11,startcol=0)
                sigma_annualisert_df.to_excel(writer,sheet_name=sheetname,startrow=19,startcol=0)
                annaualisert_covariance.to_excel(writer,sheet_name=sheetname,startrow=27,startcol=0)
        elif s=='Kostnadsanalyse':
            sheetname = s
            kostnad_df.to_excel(writer,sheet_name=sheetname,index=False)

            
#skalering av grafer til Excell ark

scaling=0.3


wb = load_workbook(output_excel_path)
for s in sheets:
    if s== 'OptimalPortefølje':
        ws = wb[s]
        graph=Image(plot_path_efficientfront)
        graph.width=graph.width*scaling
        graph.height=graph.height*scaling
        ws.add_image(graph,'E1')

    elif s=='avkastnings_analyse':
        ws = wb[s]
        ws['A1']='Avkastningserie Statestikk'
        ws['B27'] = 'Annualisert kovariansmatrise'
        hist=Image(plot_path_histogram_avkastning)
        hist.width=hist.width*scaling
        hist.height=hist.height*scaling
        ws.add_image(hist,'H1')    
        

for ws in wb.worksheets:
    ws.sheet_view.showGridLines = False

    for col_cells in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col_cells[0].column)

        for cell in col_cells:
            value = "" if cell.value is None else str(cell.value)
            max_length = max(max_length, len(value))

        ws.column_dimensions[col_letter].width = min(max_length + 2, 30)


    wb.save(output_excel_path)

    print(f"\nExcel workbook saved to: {output_excel_path}")



